# Task 6 — Context-Aware Chat Assistant with Memory
Stores the full conversation history (Buffer Memory) and injects it as context on every new question, so the assistant can refer back to things said earlier in the session.

## Imports

In [1]:
!pip install -q transformers torch accelerate

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

## Model Loading

In [3]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"  # swap for a bigger model (e.g. Qwen2.5-7B-Instruct) if you have the GPU/RAM for it

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    device_map="auto",
    dtype="auto"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model ready on {device}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready on cuda


## Memory
Conversation Buffer Memory: every (user, assistant) turn is kept in full and replayed on every new question — no summarizing, no windowing.

In [4]:
conversation_history = []  # [{"user": ..., "assistant": ...}, ...]

## Build Context For Each Prompt
Combines the stored history with the new question into Qwen's chat-message format, so `apply_chat_template` handles the exact prompt structure the model expects.

In [5]:
def build_messages(history: list, new_question: str) -> list:
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Use the conversation history to answer consistently."}
    ]

    for turn in history:
        messages.append({"role": "user", "content": turn["user"]})
        messages.append({"role": "assistant", "content": turn["assistant"]})

    messages.append({"role": "user", "content": new_question})
    return messages

## Generate Response

In [6]:
def get_response(messages: list, max_new_tokens: int = 150) -> str:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## Chat Function
Wires memory + context building + generation together, then stores the new turn back into history so the next question can see it.

In [7]:
def chat(user_message: str) -> str:
    messages = build_messages(conversation_history, user_message)
    response = get_response(messages)

    conversation_history.append({
        "user": user_message,
        "assistant": response
    })

    return response

## Helper: Print History

In [8]:
def print_history():
    print("=" * 60)
    print("FULL CONVERSATION HISTORY")
    print("=" * 60)
    for i, turn in enumerate(conversation_history, 1):
        print(f"Turn {i}:")
        print(f"  User     : {turn['user']}")
        print(f"  Assistant: {turn['assistant']}")
        print()

## Test — Observe How Memory Affects Responses
Follows the task's own example: tell the assistant a name, then ask for it back — if memory works, it answers correctly instead of saying it doesn't know.

In [9]:
def show_chat(user_msg: str):
    response = chat(user_msg)
    print(f"User     : {user_msg}")
    print(f"Assistant: {response}")
    print()

show_chat("My name is Ahmed.")
show_chat("What is my name?")
show_chat("I study Computer and Information Science at Ain Shams University.")
show_chat("What do you know about me so far?")

User     : My name is Ahmed.
Assistant: Hello Ahmed! It's nice to meet you. How can I assist you today?

User     : What is my name?
Assistant: Your name is Ahmed.

User     : I study Computer and Information Science at Ain Shams University.
Assistant: That's great! What kind of projects or interests do you have in computer science that you're working on currently?

User     : What do you know about me so far?
Assistant: Based on our previous interactions, I've learned that you are Ahmed and you are studying Computer and Information Science at Ain Shams University.



In [10]:
print_history()

FULL CONVERSATION HISTORY
Turn 1:
  User     : My name is Ahmed.
  Assistant: Hello Ahmed! It's nice to meet you. How can I assist you today?

Turn 2:
  User     : What is my name?
  Assistant: Your name is Ahmed.

Turn 3:
  User     : I study Computer and Information Science at Ain Shams University.
  Assistant: That's great! What kind of projects or interests do you have in computer science that you're working on currently?

Turn 4:
  User     : What do you know about me so far?
  Assistant: Based on our previous interactions, I've learned that you are Ahmed and you are studying Computer and Information Science at Ain Shams University.



## Run a Continuous Chat Loop
Interactive loop for manual testing. Run this cell and type messages; type `exit` to stop, `clear` to reset memory.

In [ ]:
conversation_history.clear()
print("Chat started! Type 'exit' to stop, 'clear' to reset memory.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    if user_input.lower() == "clear":
        conversation_history.clear()
        print("Memory cleared!")
        continue

    if not user_input:
        continue

    response = chat(user_input)
    print(f"Assistant: {response}\n")

Chat started! Type 'exit' to stop, 'clear' to reset memory.



You:  What do you know about me so far?


Assistant: I'm sorry, but I don't have any information about you based on our previous interaction. As an artificial intelligence language model, my responses are generated from patterns in large amounts of text data rather than personal interactions or specific user profiles. If you'd like to share some details with me, please feel free to ask questions or provide information that would help me understand your context better.



You:  my name is roqaya and my friend name is amira


Assistant: It's nice to meet you Roqaya! How can I assist you today?



You:  what's the name of my friend


Assistant: Your friend's name is Amira.

